# LLM Zoomcamp 2026 - Homework 1: Agentic RAG

This notebook solves Homework 1 step by step. The goal is to build a RAG system over the course lesson markdown files, compare full-document RAG with chunked RAG, and then make the retrieval flow agentic with PydanticAI.

## Setup

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')

openai_client = OpenAI()
MODEL = 'gpt-5.4-mini'

## Q1. Load Lesson Pages

We load only markdown files whose path contains `/lessons/` from the fixed course commit. Each parsed document has `filename` and `content`.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)

files = reader.read()
documents = [file.parse() for file in files]

len(documents)

72

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing And Searching

The full lesson document is searchable through `content`, while `filename` is kept as metadata.

In [4]:
from minsearch import Index

query = 'How does the agentic loop keep calling the model until it stops?'

index = Index(
    text_fields=['content'],
    keyword_fields=['filename'],
)
index.fit(documents)

results = index.search(query, num_results=5)
[(doc['filename'], len(doc['content'])) for doc in results]

[('01-agentic-rag/lessons/14-agentic-loop.md', 10121),
 ('01-agentic-rag/lessons/15-frameworks.md', 5379),
 ('01-agentic-rag/lessons/13-function-calling.md', 8267),
 ('01-agentic-rag/lessons/11-agents-intro.md', 1967),
 ('01-agentic-rag/lessons/16-other-frameworks.md', 3799)]

## Q3. RAG Over Full Documents

The course helper was written for FAQ documents with `section`, `question`, and `answer`. Here the schema is `filename` and `content`, so the context builder changes. We also keep the full response so we can read token usage.

In [5]:
from dataclasses import dataclass

INSTRUCTIONS = '''
Your task is to answer questions from course participants based on the
provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know."
'''.strip()

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

@dataclass
class RAGResult:
    answer: str
    usage: object
    search_results: list

class LessonRAG:
    def __init__(self, index, llm_client, model=MODEL):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            label = f"FILE: {doc['filename']}"
            if 'start' in doc:
                label = f"{label} START: {doc['start']}"
            lines.append(f"{label}\n{doc['content']}")
        return '\n\n'.join(lines)

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return PROMPT_TEMPLATE.format(question=query, context=context)

    def llm(self, prompt):
        return self.llm_client.responses.create(
            model=self.model,
            input=[
                {'role': 'developer', 'content': INSTRUCTIONS},
                {'role': 'user', 'content': prompt},
            ],
        )

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResult(
            answer=response.output_text,
            usage=response.usage,
            search_results=search_results,
        )

In [6]:
full_rag = LessonRAG(index=index, llm_client=openai_client)
full_result = full_rag.rag(query)

print(full_result.answer)
print()
print('input_tokens:', full_result.usage.input_tokens)
print('output_tokens:', full_result.usage.output_tokens)
print('total_tokens:', full_result.usage.total_tokens)

It keeps calling the model in a `while True` loop.

After each model response, the code checks whether any `function_call` items were returned:

- if there are function calls, it runs the tool, appends the tool output to `messages`, and loops again;
- if there are no function calls, it breaks out of the loop.

So the stop condition is simply: **no function calls this turn**.

input_tokens: 7123
output_tokens: 90
total_tokens: 7213


## Q4. Chunking

Chunking splits long lesson pages into overlapping windows. With size 2000 and step 1000, adjacent chunks overlap by 1000 characters.

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [8]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

## Q5. RAG With Chunking

Now we index chunks instead of full pages and run the same RAG query. This should reduce the amount of context sent to the model.

In [9]:
chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename'],
)
chunk_index.fit(chunks)

chunk_results = chunk_index.search(query, num_results=5)
[(doc['filename'], doc.get('start'), len(doc['content'])) for doc in chunk_results]

[('01-agentic-rag/lessons/14-agentic-loop.md', 4000, 2000),
 ('01-agentic-rag/lessons/14-agentic-loop.md', 5000, 2000),
 ('01-agentic-rag/lessons/14-agentic-loop.md', 0, 2000),
 ('01-agentic-rag/lessons/15-frameworks.md', 4000, 1379),
 ('01-agentic-rag/lessons/15-frameworks.md', 3000, 2000)]

In [10]:
chunk_rag = LessonRAG(index=chunk_index, llm_client=openai_client)
chunk_result = chunk_rag.rag(query)

print(chunk_result.answer)
print()
print('full input tokens:', full_result.usage.input_tokens)
print('chunk input tokens:', chunk_result.usage.input_tokens)
print('difference:', full_result.usage.input_tokens - chunk_result.usage.input_tokens)
print('ratio:', full_result.usage.input_tokens / chunk_result.usage.input_tokens)

It keeps calling the model inside a `while True` loop, and after each response it checks whether there were any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the result to `messages`, and keeps looping.
- If the model returns a message with no function calls, `has_function_calls` stays `False`, so the loop `break`s.

So the stop condition is: **no function calls in the current turn**.

full input tokens: 7123
chunk input tokens: 2330
difference: 4793
ratio: 3.057081545064378


## Q6. Agentic RAG With PydanticAI

For the agentic version, the model gets a `search` tool. The function increments `search_calls` every time it runs, which gives us a direct count of tool calls.

In [11]:
from pydantic_ai import Agent

agent_instructions = '''
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
'''.strip()

agent_question = 'How does the agentic loop work, and how is it different from plain RAG?'

search_calls = []

agent = Agent(
    'openai-chat:gpt-5.4-mini',
    instructions=agent_instructions,
)

@agent.tool_plain
def search(query: str) -> str:
    '''
    Search the course lesson chunks for passages matching the given query.
    '''
    search_calls.append(query)
    results = chunk_index.search(query, num_results=5)

    formatted = []
    for doc in results:
        formatted.append(
            f"FILE: {doc['filename']} START: {doc.get('start')}\n"
            f"{doc['content']}"
        )

    return '\n\n'.join(formatted)

agent_result = await agent.run(agent_question)

print('tool calls:', len(search_calls))
for i, call_query in enumerate(search_calls, start=1):
    print(i, call_query)
print()
print(agent_result.output)

tool calls: 3
1 agentic loop plain RAG difference lesson
2 agentic loop iterative retrieval planning action observation
3 RAG retrieve generate one-shot vs agentic loop course

The **agentic loop** is a repeated cycle where the **LLM decides what to do next**, your code executes that action, and the result is sent back to the LLM. In the course, it’s described as a `while True` loop that:

1. sends the current messages to the model,
2. checks whether the model returned any tool/function calls,
3. executes those tool calls,
4. appends the tool outputs to the conversation,
5. repeats until the model returns a normal message with **no more function calls**.

So the key idea is: **the model is in control of the sequence**. The loop stops when there are no more tool calls. The course also notes that frameworks wrap this same pattern and add things like max-iteration limits, token budgets, and chat history management.

### How it differs from plain RAG

**Plain RAG** is a fixed pipeline:

``

## Final Answers

Based on this run:

- Q1: 72
- Q2: `01-agentic-rag/lessons/14-agentic-loop.md`
- Q3: 7000
- Q4: 295
- Q5: 3x fewer
- Q6: 4 (closest option; this run made 3 tool calls)

Q6 can vary slightly because the model decides how many searches to make.
